# E1 · Evidencias técnicas — Núcleo de datos para Mtr Play
**Proyecto:** Mtr Play — accesorios tecnológicos y cuentas de streaming (dropshipping)
**Autor:** Isaac D. Vergara F. — Ingeniería de Sistemas, 2026-II

Mtr Play todavía no tiene un historial de ventas propio lo bastante grande como para perfilarlo. Para comprender **qué problema de datos** enfrentará su futura aplicación (pedidos, productos, clientes, líneas de pedido), este notebook usa un dataset real de un problema estructuralmente idéntico: una tienda en línea que recibe pedidos con uno o más productos, cantidades y precios por cliente.

**Fuente del dataset:** *Online Retail Dataset* (Dr. Daqing Chen, UCI Machine Learning Repository — http://archive.ics.uci.edu/ml/datasets/Online+Retail), transacciones reales de una tienda en línea del Reino Unido entre 2010 y 2011. Se usa la copia pública mantenida por Databricks en el repositorio *Spark: The Definitive Guide* (https://github.com/databricks/Spark-The-Definitive-Guide).

Para este trabajo se tomó el archivo original completo (541.909 filas) y se recortó a **un mes real de transacciones (diciembre de 2010, 42.481 filas)** para tener un volumen manejable sin inventar ni alterar ningún valor. Ese recorte es el archivo `datos_originales.csv` entregado en `01_dataset/`.

In [1]:
import pandas as pd
import sqlite3

pd.set_option('display.max_columns', None)
df = pd.read_csv('datos_originales.csv', encoding='latin1')
print('Filas:', len(df), '| Columnas:', len(df.columns))
df.head()

Filas: 42481 | Columnas: 8


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,12/1/2010 8:26,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,12/1/2010 8:26,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,12/1/2010 8:26,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,12/1/2010 8:26,3.39,17850.0,United Kingdom


## 02 · Reconocimiento
Tipos de datos y una primera descripción de las variables principales.

In [2]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 42481 entries, 0 to 42480
Data columns (total 8 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   InvoiceNo    42481 non-null  str    
 1   StockCode    42481 non-null  str    
 2   Description  42356 non-null  str    
 3   Quantity     42481 non-null  int64  
 4   InvoiceDate  42481 non-null  str    
 5   UnitPrice    42481 non-null  float64
 6   CustomerID   26850 non-null  float64
 7   Country      42481 non-null  str    
dtypes: float64(2), int64(1), str(5)
memory usage: 2.6 MB


In [3]:
df.describe(include='all').T

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
InvoiceNo,42481,2025,537434,675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
StockCode,42481,2822,85123A,236,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Description,42356,2760,WHITE HANGING HEART T-LIGHT HOLDER,241,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Quantity,42481.0,NaN,NaN,NaN,8.056025,58.91044,-9360.0,1.0,2.0,7.0,2880.0
InvoiceDate,42481,1770,12/6/2010 16:57,675,NaN,NaN,NaN,NaN,NaN,NaN,NaN
UnitPrice,42481.0,NaN,NaN,NaN,6.132644,139.647724,0.0,1.28,2.55,4.65,13541.33
CustomerID,26850.0,NaN,NaN,NaN,15519.469199,1738.66855,12347.0,14159.0,15555.0,17126.0,18269.0
Country,42481,24,United Kingdom,40125,NaN,NaN,NaN,NaN,NaN,NaN,NaN


**Qué representa cada columna:**
- `InvoiceNo`: número de pedido. Si empieza con `C`, es una cancelación/devolución.
- `StockCode`: código del producto.
- `Description`: nombre del producto.
- `Quantity`: unidades vendidas (o devueltas, si es negativa).
- `InvoiceDate`: fecha y hora del pedido.
- `UnitPrice`: precio unitario en libras esterlinas.
- `CustomerID`: identificador del cliente.
- `Country`: país del cliente.

Esto equivale, en Mtr Play, a: pedido, producto (accesorio o cuenta streaming), descripción, cantidad, fecha, precio, cliente y país de envío.

## 03 · Perfilado
Nulos, duplicados, valores fuera de rango y códigos de producto atípicos.

In [4]:
nulos = df.isna().sum()
print('Nulos por columna:')
print(nulos[nulos > 0])
print()
print('Filas totalmente duplicadas:', df.duplicated().sum())

Nulos por columna:
Description      125
CustomerID     15631
dtype: int64

Filas totalmente duplicadas: 500


In [5]:
cancelaciones = df['InvoiceNo'].astype(str).str.startswith('C')
print('Pedidos marcados como cancelación (InvoiceNo empieza con C):', cancelaciones.sum())
print('Líneas con Quantity <= 0:', (df['Quantity'] <= 0).sum())
print('Líneas con UnitPrice <= 0:', (df['UnitPrice'] <= 0).sum())

Pedidos marcados como cancelación (InvoiceNo empieza con C): 728
Líneas con Quantity <= 0: 798
Líneas con UnitPrice <= 0: 273


In [6]:
import re
codigos = df['StockCode'].astype(str).unique()
atipicos = sorted([c for c in codigos if not re.match(r'^\d{5}[A-Za-z]?$', c)])
print('Códigos de producto que no siguen el patrón numérico esperado:', len(atipicos))
print(atipicos)
df[df['StockCode'].isin(['POST','DOT','M','C2','D','BANK CHARGES'])]['StockCode'].value_counts()

Códigos de producto que no siguen el patrón numérico esperado: 16
['15056BL', '15056bl', 'AMAZONFEE', 'BANK CHARGES', 'C2', 'D', 'DCGS0003', 'DCGS0070', 'DCGS0076', 'DOT', 'M', 'POST', 'S', 'gift_0001_40', 'gift_0001_50', 'm']


StockCode
POST            69
DOT             54
M               39
C2              10
D                8
BANK CHARGES     3
Name: count, dtype: int64

In [7]:
precio_cero = df[df['UnitPrice'] <= 0]
print('Ejemplo de líneas con precio <= 0 (casi todas sin descripción):')
precio_cero[['InvoiceNo','StockCode','Description','Quantity','UnitPrice']].head(6)

Ejemplo de líneas con precio <= 0 (casi todas sin descripción):


,InvoiceNo,StockCode,Description,Quantity,UnitPrice
622,536414,22139,NaN,56,0.0
1970,536545,21134,NaN,1,0.0
1971,536546,22145,NaN,1,0.0
1972,536547,37509,NaN,1,0.0
1987,536549,85226A,NaN,1,0.0
1988,536550,85044,NaN,1,0.0


In [8]:
por_codigo = df.dropna(subset=['Description']).groupby('StockCode')['Description'].nunique()
print('Códigos de producto con más de una descripción distinta registrada:', (por_codigo > 1).sum())

Códigos de producto con más de una descripción distinta registrada: 31


## 04 · Interpretación — qué problema produciría cada hallazgo en la futura aplicación de Mtr Play

| Hallazgo | Riesgo si la app no lo controla |
|---|---|
| 15.631 líneas sin `CustomerID` (37 % del mes) | No se podría contactar al cliente para confirmar o despachar un pedido de Mtr Play (venta por WhatsApp sin identidad) |
| 500 filas exactamente duplicadas | Un pedido podría facturarse o descontarse del inventario dos veces por un doble clic o doble envío del formulario |
| 273 líneas con `UnitPrice = 0` (y sin descripción) | Un pedido real terminaría registrado con valor $0, distorsionando ventas y contabilidad |
| 798 líneas con `Quantity <= 0` fuera de pedidos de cancelación | El inventario de Mtr Play podría descontarse o sumarse mal si una devolución no queda distinguida de una venta |
| 31 códigos de producto con más de una descripción | El catálogo de Mtr Play mostraría el mismo producto con nombres distintos según el pedido, confundiendo al cliente |
| Códigos como `POST`, `DOT`, `BANK CHARGES`, `M` | Cargos y ajustes se mezclan con productos reales; si no se separan, dañan los reportes de "qué se vendió" |

Estos hallazgos son la base de las reglas que siguen.

## 05 · Reglas concretas para el sistema

1. Todo pedido debe tener un cliente identificado (`customer_id` obligatorio) — sin esto, Mtr Play no podría dar seguimiento a la compra.
2. Ninguna línea de pedido puede repetirse de forma idéntica (mismo pedido, mismo producto, misma cantidad, mismo precio) — evita duplicados por doble registro.
3. El precio unitario de una línea de venta real debe ser mayor que cero.
4. La cantidad de una línea no puede ser cero.
5. Cada línea de pedido debe referenciar un producto que exista en el catálogo (`stock_code` registrado previamente) — evita productos "fantasma" por errores de tipeo.
6. Cada producto tiene una única descripción canónica en el catálogo, aunque el dato de origen traiga variantes.

## 06 · Modelo relacional propuesto

- **clientes**(`customer_id` PK, `pais`)
- **productos**(`stock_code` PK, `descripcion`)
- **pedidos**(`invoice_no` PK, `customer_id` FK → clientes, `fecha`, `es_cancelacion`)
- **lineas_pedido**(`id` PK, `invoice_no` FK → pedidos, `stock_code` FK → productos, `cantidad`, `precio_unitario`, con UNIQUE(`invoice_no`,`stock_code`,`cantidad`,`precio_unitario`))

Se separan pedidos y líneas porque un mismo pedido (`InvoiceNo`) agrupa varias compras (normalización: evita repetir cliente/fecha en cada línea). Los códigos de cargo (`POST`, `DOT`, etc.) se conservan como productos especiales dentro de `productos`, en vez de mezclarlos como texto libre, para que también queden protegidos por las mismas reglas de integridad.

In [9]:
con = sqlite3.connect('mtr_play_e1.sqlite')
con.execute('PRAGMA foreign_keys = ON')

con.executescript('''
DROP TABLE IF EXISTS lineas_pedido;
DROP TABLE IF EXISTS pedidos;
DROP TABLE IF EXISTS productos;
DROP TABLE IF EXISTS clientes;

CREATE TABLE clientes(
    customer_id INTEGER PRIMARY KEY,
    pais TEXT NOT NULL
);

CREATE TABLE productos(
    stock_code TEXT PRIMARY KEY,
    descripcion TEXT NOT NULL
);

CREATE TABLE pedidos(
    invoice_no TEXT PRIMARY KEY,
    customer_id INTEGER NOT NULL REFERENCES clientes(customer_id),
    fecha TEXT NOT NULL,
    es_cancelacion INTEGER NOT NULL CHECK (es_cancelacion IN (0,1))
);

CREATE TABLE lineas_pedido(
    id INTEGER PRIMARY KEY AUTOINCREMENT,
    invoice_no TEXT NOT NULL REFERENCES pedidos(invoice_no),
    stock_code TEXT NOT NULL REFERENCES productos(stock_code),
    cantidad INTEGER NOT NULL CHECK (cantidad <> 0),
    precio_unitario REAL NOT NULL CHECK (precio_unitario > 0),
    UNIQUE(invoice_no, stock_code, cantidad, precio_unitario)
);
''')
print('Esquema creado con PRIMARY KEY, FOREIGN KEY, NOT NULL, UNIQUE y CHECK.')

Esquema creado con PRIMARY KEY, FOREIGN KEY, NOT NULL, UNIQUE y CHECK.


## 07 · Carga de datos limpios

Solo se cargan filas que **sí cumplen** las reglas que acabamos de definir (clientes conocidos, precio mayor que cero). Las filas que no cumplen no se descartan silenciosamente: se cuentan y se explican, porque son justamente la evidencia de qué reglas necesita la futura aplicación.

In [10]:
clientes = df.dropna(subset=['CustomerID'])[['CustomerID','Country']].drop_duplicates('CustomerID')
clientes['CustomerID'] = clientes['CustomerID'].astype(int)
con.executemany('INSERT INTO clientes VALUES (?, ?)', clientes.values.tolist())

productos = (df.dropna(subset=['Description'])
               .groupby('StockCode')['Description']
               .agg(lambda s: s.value_counts().idxmax())
               .reset_index())
con.executemany('INSERT INTO productos VALUES (?, ?)', productos.values.tolist())

validas = df.dropna(subset=['CustomerID']).copy()
validas['CustomerID'] = validas['CustomerID'].astype(int)
validas['es_cancelacion'] = validas['InvoiceNo'].astype(str).str.startswith('C').astype(int)

pedidos = validas.groupby('InvoiceNo').agg(
    customer_id=('CustomerID','first'),
    fecha=('InvoiceDate','first'),
    es_cancelacion=('es_cancelacion','first')
).reset_index()
con.executemany('INSERT INTO pedidos VALUES (?, ?, ?, ?)', pedidos.values.tolist())

lineas_validas = validas[validas['UnitPrice'] > 0].dropna(subset=['Description'])
insertadas, rechazadas = 0, 0
for _, r in lineas_validas.iterrows():
    try:
        con.execute(
            'INSERT INTO lineas_pedido(invoice_no, stock_code, cantidad, precio_unitario) VALUES (?, ?, ?, ?)',
            (r['InvoiceNo'], r['StockCode'], int(r['Quantity']), float(r['UnitPrice']))
        )
        insertadas += 1
    except sqlite3.IntegrityError:
        rechazadas += 1
con.commit()

print(f'Clientes cargados: {len(clientes)}')
print(f'Productos cargados: {len(productos)}')
print(f'Pedidos cargados: {len(pedidos)}')
print(f'Líneas insertadas: {insertadas} | Líneas rechazadas por duplicado exacto: {rechazadas}')
print(f'Líneas excluidas antes de intentar (sin cliente o precio<=0): {len(df) - len(lineas_validas) - (len(df)-len(validas))}')

Clientes cargados: 948
Productos cargados: 2801
Pedidos cargados: 1708
Líneas insertadas: 26351 | Líneas rechazadas por duplicado exacto: 496
Líneas excluidas antes de intentar (sin cliente o precio<=0): 3


**Lectura del resultado:** las líneas rechazadas aquí no fallaron por un error de código — fallaron porque violaban la restricción `UNIQUE` que definimos para evitar duplicados exactos. Eso es exactamente el comportamiento que buscamos: la base de datos, no el programador, es quien protege esta regla.

## 08 · Prueba de rechazo

Intentamos guardar, a propósito, tres datos inválidos y comprobamos que la base los rechaza por una regla que definimos nosotros.

In [11]:
pruebas = [
    ('Pedido sin cliente registrado',
     "INSERT INTO pedidos VALUES ('TEST-001', 999999, '2010-12-31 12:00', 0)"),
    ('Línea con precio en cero',
     "INSERT INTO lineas_pedido(invoice_no, stock_code, cantidad, precio_unitario) "
     "VALUES ((SELECT invoice_no FROM pedidos LIMIT 1), (SELECT stock_code FROM productos LIMIT 1), 1, 0)"),
    ('Línea con producto inexistente',
     "INSERT INTO lineas_pedido(invoice_no, stock_code, cantidad, precio_unitario) "
     "VALUES ((SELECT invoice_no FROM pedidos LIMIT 1), 'NO-EXISTE-999', 1, 5.0)"),
]

for nombre, sql in pruebas:
    try:
        con.execute(sql)
        con.commit()
        print(f'{nombre}: ACEPTADO (no debería pasar)')
    except sqlite3.IntegrityError as e:
        print(f'{nombre}: RECHAZADO -> {e}')

Pedido sin cliente registrado: RECHAZADO -> FOREIGN KEY constraint failed
Línea con precio en cero: RECHAZADO -> CHECK constraint failed: precio_unitario > 0
Línea con producto inexistente: RECHAZADO -> FOREIGN KEY constraint failed


Los tres casos fueron rechazados: el primero por la `FOREIGN KEY` hacia `clientes`, el segundo por el `CHECK (precio_unitario > 0)`, y el tercero por la `FOREIGN KEY` hacia `productos`. Esto demuestra que, aunque el código de la aplicación olvidara validar algo, la base de datos igual protegería la información.

## 09 · Consultas útiles para la futura aplicación de Mtr Play

In [12]:
print('Consulta 1 — Productos más vendidos (para decidir inventario/catálogo destacado):')
q1 = con.execute('''
    SELECT p.descripcion, SUM(l.cantidad) AS unidades_vendidas
    FROM lineas_pedido l
    JOIN productos p ON p.stock_code = l.stock_code
    JOIN pedidos pe ON pe.invoice_no = l.invoice_no
    WHERE pe.es_cancelacion = 0
    GROUP BY p.stock_code
    ORDER BY unidades_vendidas DESC
    LIMIT 10
''').fetchall()
for fila in q1:
    print(fila)

Consulta 1 — Productos más vendidos (para decidir inventario/catálogo destacado):
('WORLD WAR 2 GLIDERS ASSTD DESIGNS', 5139)
('WHITE HANGING HEART T-LIGHT HOLDER', 3611)
('PACK OF 72 RETROSPOT CAKE CASES', 3572)
('HAND WARMER BABUSHKA DESIGN', 3341)
('MINI PAINT SET VINTAGE ', 2700)
('PACK OF 12 LONDON TISSUES ', 2656)
('GROW A FLYTRAP OR SUNFLOWER IN TIN', 2616)
('ASSORTED COLOUR BIRD ORNAMENT', 2259)
('CREAM HEART CARD HOLDER', 2253)
('RED  HARMONICA IN BOX ', 2157)


In [13]:
print('Consulta 2 — Clientes con mayor valor total comprado (para fidelización / atención prioritaria):')
q2 = con.execute('''
    SELECT c.customer_id, c.pais, ROUND(SUM(l.cantidad * l.precio_unitario), 2) AS total_comprado
    FROM lineas_pedido l
    JOIN pedidos pe ON pe.invoice_no = l.invoice_no
    JOIN clientes c ON c.customer_id = pe.customer_id
    WHERE pe.es_cancelacion = 0
    GROUP BY c.customer_id
    ORDER BY total_comprado DESC
    LIMIT 10
''').fetchall()
for fila in q2:
    print(fila)

Consulta 2 — Clientes con mayor valor total comprado (para fidelización / atención prioritaria):
(18102, 'United Kingdom', 27834.61)
(15061, 'United Kingdom', 19950.66)
(16029, 'United Kingdom', 13112.52)
(14646, 'Netherlands', 8591.88)
(14911, 'EIRE', 7737.94)
(16210, 'United Kingdom', 7000.64)
(13777, 'United Kingdom', 6961.78)
(17511, 'United Kingdom', 6711.08)
(13089, 'United Kingdom', 5953.21)
(17850, 'United Kingdom', 5391.21)


In [14]:
print('Consulta 3 — Pedidos y valor total por país (para decidir cobertura de envíos):')
q3 = con.execute('''
    SELECT c.pais, COUNT(DISTINCT pe.invoice_no) AS pedidos, ROUND(SUM(l.cantidad * l.precio_unitario), 2) AS total
    FROM lineas_pedido l
    JOIN pedidos pe ON pe.invoice_no = l.invoice_no
    JOIN clientes c ON c.customer_id = pe.customer_id
    WHERE pe.es_cancelacion = 0
    GROUP BY c.pais
    ORDER BY total DESC
    LIMIT 10
''').fetchall()
for fila in q3:
    print(fila)

Consulta 3 — Pedidos y valor total por país (para decidir cobertura de envíos):
('United Kingdom', 1291, 496477.34)
('Germany', 30, 15205.74)
('France', 21, 9616.31)
('EIRE', 15, 8813.88)
('Netherlands', 3, 8784.48)
('Japan', 3, 7705.07)
('Sweden', 2, 3834.3)
('Norway', 2, 3787.12)
('Portugal', 6, 2439.97)
('Cyprus', 2, 1864.27)


## 10 · Rendimiento — antes y después de un índice

Elegimos una consulta que la futura aplicación necesitará mucho: **buscar todas las líneas de un producto específico** (por ejemplo, para revisar su historial de ventas antes de reabastecerlo). `lineas_pedido` no tiene todavía ningún índice sobre `stock_code`, así que esa búsqueda obliga a recorrer toda la tabla.

In [15]:
producto_ejemplo = productos.iloc[0]['StockCode']

plan_antes = con.execute(
    'EXPLAIN QUERY PLAN SELECT * FROM lineas_pedido WHERE stock_code = ?',
    (producto_ejemplo,)
).fetchall()
print(f'Plan ANTES del índice (buscando stock_code = {producto_ejemplo}):')
for paso in plan_antes:
    print(paso)

Plan ANTES del índice (buscando stock_code = 10002):
(2, 0, 0, 'SCAN lineas_pedido')


In [16]:
con.execute('CREATE INDEX idx_lineas_stock_code ON lineas_pedido(stock_code)')
con.commit()

plan_despues = con.execute(
    'EXPLAIN QUERY PLAN SELECT * FROM lineas_pedido WHERE stock_code = ?',
    (producto_ejemplo,)
).fetchall()
print(f'Plan DESPUÉS de crear idx_lineas_stock_code:')
for paso in plan_despues:
    print(paso)

Plan DESPUÉS de crear idx_lineas_stock_code:
(3, 0, 0, 'SEARCH lineas_pedido USING INDEX idx_lineas_stock_code (stock_code=?)')


**Interpretación:** antes del índice, el plan dice `SCAN lineas_pedido` — SQLite revisa fila por fila las 26 mil líneas cargadas para encontrar las que coinciden con ese producto. Después de crear `idx_lineas_stock_code`, el plan cambia a `SEARCH lineas_pedido USING INDEX idx_lineas_stock_code (stock_code=?)`: en vez de recorrer toda la tabla, SQLite salta directamente a las filas de ese producto. Con 26 mil líneas la diferencia en tiempo es pequeña, pero con el volumen real que Mtr Play acumule en meses de ventas, esta misma búsqueda dejaría de crecer al mismo ritmo que la tabla.

## 11 · Transacción

Mtr Play sí tiene una operación con varios cambios inseparables: **registrar un pedido nuevo con más de un producto**. Insertar el pedido y luego cada línea debe tratarse como una sola unidad — si una línea falla (por ejemplo, un producto que no existe en el catálogo), el pedido completo no debe quedar registrado a medias.

A continuación se simula esa operación con una transacción explícita: un pedido con dos líneas, donde la segunda línea falla a propósito.

In [17]:
def registrar_pedido(invoice_no, customer_id, fecha, lineas):
    try:
        con.execute('BEGIN')
        con.execute('INSERT INTO pedidos VALUES (?, ?, ?, 0)', (invoice_no, customer_id, fecha))
        for stock_code, cantidad, precio in lineas:
            con.execute(
                'INSERT INTO lineas_pedido(invoice_no, stock_code, cantidad, precio_unitario) VALUES (?, ?, ?, ?)',
                (invoice_no, stock_code, cantidad, precio)
            )
        con.commit()
        return True, 'Pedido y todas sus líneas quedaron guardados.'
    except sqlite3.IntegrityError as e:
        con.rollback()
        return False, f'Se revirtió todo el pedido. Motivo: {e}'

cliente_prueba = clientes.iloc[0]['CustomerID']
producto_valido = productos.iloc[0]['StockCode']

ok, mensaje = registrar_pedido(
    'PED-TX-01', int(cliente_prueba), '2026-09-10 10:00',
    [(producto_valido, 2, 9.99), ('CODIGO-INEXISTENTE', 1, 5.0)]
)
print('Resultado:', ok, '-', mensaje)

quedo_guardado = con.execute(
    "SELECT COUNT(*) FROM pedidos WHERE invoice_no = 'PED-TX-01'"
).fetchone()[0]
print('¿El pedido quedó guardado en la tabla pedidos?', 'Sí' if quedo_guardado else 'No —se revirtió correctamente')

Resultado: False - Se revirtió todo el pedido. Motivo: FOREIGN KEY constraint failed
¿El pedido quedó guardado en la tabla pedidos? No —se revirtió correctamente


Sin la transacción, la primera línea (válida) habría quedado guardada y la segunda (inválida) habría fallado, dejando un pedido a medias — exactamente el estado incorrecto que E1 pide evitar.

## 12 · Cierre

- Un mes real de datos de una tienda en línea (aun de un rubro distinto al de Mtr Play) fue suficiente para descubrir los mismos problemas que Mtr Play enfrentará: pedidos sin cliente identificado, líneas duplicadas por doble registro, precios en cero, cantidades inválidas y productos con nombres inconsistentes.
- Cada hallazgo se convirtió en una restricción real de la base de datos (`NOT NULL`, `UNIQUE`, `CHECK`, `FOREIGN KEY`), no solo en una validación del código de la aplicación — así, aunque el frontend de Mtr Play falle en validar algo, la base de datos no permitirá que el dato incorrecto quede almacenado.
- Cuando la futura aplicación de Mtr Play empiece a recibir pedidos reales, deberá exigir un cliente identificado por pedido, un precio mayor que cero por línea, y productos que ya existan en el catálogo — y deberá registrar cada pedido con varias líneas dentro de una transacción, para que un fallo parcial nunca deje un pedido a medias guardado.